In [1]:
import json, time
from pathlib import Path
from collections import Counter

import torch
from PIL import Image
from tqdm import tqdm
from transformers import Blip2Processor, Blip2ForConditionalGeneration

/home/ubuntu/miniforge3/envs/register_work/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 123
MODEL_ID = "Salesforce/blip2-flan-t5-xl"

# Point this at your balanced 5k VQAv2 subset metadata
# Example: /home/ubuntu/neel/subsets/vqav2_balanced_5k/metadata.jsonl
META_PATH = Path("subsets/vqa_v2_balanced_5k/metadata.jsonl")

OUT_DIR = Path("runs/blip2/vqa/vqa_v2_balanced_5k")
PRED_PATH = OUT_DIR / "preds.jsonl"
CFG_PATH = OUT_DIR / "config.json"

In [3]:
def read_jsonl(path: Path):
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)

def append_jsonl(path: Path, row: dict):
    with open(path, "a") as f:
        f.write(json.dumps(row) + "\n")

def write_json(path: Path, obj: dict):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

def count_nonempty_lines(path: Path) -> int:
    if not path.exists():
        return 0
    n = 0
    with open(path, "r") as f:
        for line in f:
            if line.strip():
                n += 1
    return n

In [4]:
def normalize_answer(s: str) -> str:
    # minimal normalization; VQAv2 eval is more involved, but this is a decent baseline
    return " ".join(s.lower().strip().split())

def extract_gt_answers(ex: dict):
    """
    Returns a list of ground-truth answers (strings) if present,
    else returns [multiple_choice_answer] if present,
    else [].
    """
    ans = ex.get("answers", None)
    if isinstance(ans, list) and len(ans) > 0:
        # answers may be list[str] or list[{"answer": str}, ...]
        if isinstance(ans[0], dict) and "answer" in ans[0]:
            return [a.get("answer", "") for a in ans if a.get("answer", "")]
        if isinstance(ans[0], str):
            return ans
    mca = ex.get("multiple_choice_answer", None)
    if isinstance(mca, str) and mca.strip():
        return [mca]
    return []

def vqav2_soft_accuracy(pred: str, gt_answers: list[str]) -> float:
    """
    VQAv2 soft accuracy: min(1, (#humans that said pred)/3)
    If only one GT answer is available, returns exact match (0/1).
    """
    if not gt_answers:
        return 0.0
    p = normalize_answer(pred)
    gts = [normalize_answer(a) for a in gt_answers if isinstance(a, str) and a.strip()]

    if len(gts) <= 1:
        return 1.0 if gts and p == gts[0] else 0.0

    c = Counter(gts)
    return min(1.0, c.get(p, 0) / 3.0)

In [5]:
def main():
    if not META_PATH.exists():
        raise FileNotFoundError(f"Missing: {META_PATH}")

    torch.manual_seed(SEED)

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available. This baseline is intended to run on the GPU.")

    device = torch.device("cuda")
    dtype = torch.float16

    print(f"Loading BLIP-2 on GPU: {MODEL_ID}")
    processor = Blip2Processor.from_pretrained(MODEL_ID)
    model = Blip2ForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    # Optional speed on A100
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    gen = {
        "max_new_tokens": 10,   # VQA answers should be short
        "num_beams": 5,
        "do_sample": False,
    }

    OUT_DIR.mkdir(parents=True, exist_ok=True)
    write_json(CFG_PATH, {
        "run_name": "blip2_vqa_vqav2_balanced_5k",
        "model_id": MODEL_ID,
        "task": "vqa",
        "dataset": "vqav2_balanced_5k",
        "seed": SEED,
        "metadata_path": str(META_PATH),
        "preds_path": str(PRED_PATH),
        "device": "cuda",
        "dtype": str(dtype),
        "gen": gen,
        "time_unix": time.time(),
    })

    rows = list(read_jsonl(META_PATH))
    total = len(rows)
    if total != 5000:
        print(f"Warning: expected 5000 metadata rows, found {total}")

    done = count_nonempty_lines(PRED_PATH)
    if done > 0:
        print(f"Resuming: {done} already written -> {PRED_PATH}")

    running_acc = 0.0
    running_n = 0

    for i in tqdm(range(done, total), desc="BLIP-2 VQA (VQAv2)"):
        ex = rows[i]

        qid = ex.get("question_id", i)
        image_file = ex["image_file"]
        question = ex["question"]

        # Prompting format for BLIP2 FLAN-T5
        prompt = f"Question: {question}\nAnswer:"

        img = Image.open(image_file).convert("RGB")
        inputs = processor(images=img, text=prompt, return_tensors="pt")
        inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}

        with torch.inference_mode():
            out_ids = model.generate(**inputs, **gen)

        pred = processor.tokenizer.decode(out_ids[0], skip_special_tokens=True).strip()

        gt_answers = extract_gt_answers(ex)
        acc = vqav2_soft_accuracy(pred, gt_answers)

        running_acc += acc
        running_n += 1
        avg_acc = running_acc / max(1, running_n)

        out_row = {
            "example_id": f"vqav2:{qid}",
            "dataset": "vqav2_balanced_5k",
            "task": "vqa",
            "question_id": qid,
            "image_file": image_file,
            "question": question,
            "gt_answers": gt_answers,
            "prediction": pred,
            "soft_acc": acc,
            "running_avg_soft_acc": avg_acc,
            "model_id": MODEL_ID,
            "seed": SEED,
            "gen": gen,
            "intervention": {"name": "none"},
        }
        append_jsonl(PRED_PATH, out_row)

    print("DONE")
    print("Predictions:", PRED_PATH)
    print("Config:", CFG_PATH)


if __name__ == "__main__":
    main()

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading BLIP-2 on GPU: Salesforce/blip2-flan-t5-xl


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:03<00:00,  1.59s/it]


Resuming: 1 already written -> runs/blip2/vqa/vqa_v2_balanced_5k/preds.jsonl


BLIP-2 VQA (VQAv2): 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4999/4999 [30:30<00:00,  2.73it/s]

DONE
Predictions: runs/blip2/vqa/vqa_v2_balanced_5k/preds.jsonl
Config: runs/blip2/vqa/vqa_v2_balanced_5k/config.json
